# Fiddler Golden Datasets Quick Start

## Goal

Build a **golden dataset** from real production traffic.

A golden dataset is a curated, stable set of test cases that every experiment runs against. The
strongest ones are not invented — they are promoted from traffic your application has actually
served: the slow responses, the ones a guardrail flagged, the ones a user complained about.

This notebook walks through promoting production spans into a dataset so you can replay them
against every future change. Each promoted span becomes a dataset item, which turns a bug you saw
once in production into a permanent regression test.

### What you'll do

1. Connect to Fiddler (creating the project and application if needed)
2. Optionally seed synthetic spans, if you don't have production traffic yet
3. Discover which attribute keys your spans actually carry
4. Query for the spans worth promoting
5. Map span attributes onto dataset fields
6. Add the spans as dataset items
7. Run an experiment against your new golden dataset

### Prerequisites

- A Fiddler API key (**Settings > Credentials**)
- Python 3.10+

You do **not** need an existing project or application — both are created if missing. If your
application has no traffic yet, an optional seeding section emits synthetic spans so the notebook
runs end to end.

## 0. Imports and Configuration

In [ ]:
# Install the Fiddler Evaluations SDK
%pip install -q --upgrade fiddler-evals

# Core imports
import json
from datetime import datetime, timedelta, timezone

import requests

# Fiddler Evaluations SDK
from fiddler_evals import (
    Application,
    Dataset,
    FieldMapping,
    OperatorType,
    Project,
    QueryCondition,
    QueryRule,
    SearchFilter,
    SearchScope,
    SpanReference,
    __version__,
    init,
)

print(f'Fiddler Evals SDK version: {__version__}')

## 1. Connect to Fiddler

**What you need:**

1. **Fiddler URL** — your instance URL (e.g. `https://your-org.fiddler.ai`)
2. **API Key** — found in the **Credentials** tab on your Fiddler **Settings** page

The project and application are created if they don't already exist, so this notebook runs whether
or not you have set them up before.

In [ ]:
# Replace with your Fiddler instance details
URL = ''  # Full URL including https:// (e.g. 'https://your_company_name.fiddler.ai')
API_KEY = ''  # Your Fiddler API key from Settings > Credentials

**Point at the application whose traffic you want to promote:**

In [ ]:
# Project + application to promote spans from.
# Both are created if they do not already exist.
PROJECT_NAME = 'golden_dataset_demo'
APPLICATION_NAME = 'support-agent'

# The golden dataset to build (created if it does not exist)
DATASET_NAME = 'support-agent-golden'

# How far back to look for spans
LOOKBACK_DAYS = 7

In [ ]:
init(url=URL, token=API_KEY)

project = Project.get_or_create(name=PROJECT_NAME)
application = Application.get_or_create(name=APPLICATION_NAME, project_id=project.id)

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=LOOKBACK_DAYS)

print(f'Project:     {project.name} ({project.id})')
print(f'Application: {application.name} ({application.id})')
print(f'Window:      {start_time.isoformat()} -> {end_time.isoformat()}')

## 1.5 Don't have spans yet? Seed some (optional)

**Skip this section if your application is already receiving production traces.**

Golden datasets are built from real traffic, so the rest of this notebook needs spans to promote.
If you just created the application above — or you want a deterministic demo — run this section to
emit a handful of synthetic support-agent traces.

The seeded spans use the exact attribute keys referenced in the field mapping later on, so
Steps 3–5 work without editing anything.

In [ ]:
# Set to False if your application already has production traffic.
SEED_SYNTHETIC_SPANS = True

In [ ]:
if SEED_SYNTHETIC_SPANS:
    from fiddler_otel import FiddlerClient

    otel_client = FiddlerClient(
        application_id=str(application.id),
        api_key=API_KEY,
        url=URL,
    )

    import time as _t

    # (user query, response, slow?) — a few slow ones so the filter demo in 2.1
    # has something to select. "slow" spans sleep past the 2s threshold.
    CONVERSATIONS = [
        ('How do I reset my password?',
         'Use the Forgot Password link on the sign-in page.', False),
        ('What is your refund policy?',
         'Refunds are available within 30 days of purchase.', True),
        ('Do you support SSO?',
         'Yes, SAML and OIDC are both supported.', False),
        ('Can I export my data?',
         'Yes, use Settings > Export to download a CSV.', False),
        ('How do I request a refund for last month?',
         'Contact billing and we will process the refund within 5 days.', True),
        ('Why was my card declined?',
         'Declines usually come from the issuing bank; try another card.', False),
        ('Is there an API rate limit?',
         'Yes, 1000 requests per minute per API key.', False),
        ('How do I cancel my plan?',
         'Cancel any time from Settings > Billing.', True),
    ]

    for user_query, llm_response, is_slow in CONVERSATIONS:
        # as_type='generation' sets fiddler.span.type = 'llm', which the
        # Span::span_type filter in section 2.1 selects on.
        with otel_client.start_as_current_span('support_agent', as_type='generation') as span:
            # set_input/set_output write gen_ai.llm.input.user and gen_ai.llm.output,
            # the semantic-convention keys Fiddler indexes for full-text search.
            # The Step 4 field mapping reads these same keys.
            span.set_input(user_query)
            span.set_model('openai/gpt-4o-mini')

            if is_slow:
                _t.sleep(2.5)  # push duration past the 2s filter threshold

            span.set_output(llm_response)

    otel_client.get_tracer_provider().force_flush()
    slow_count = sum(1 for _, _, slow in CONVERSATIONS if slow)
    print(f'Emitted {len(CONVERSATIONS)} spans ({slow_count} slower than 2s) '
          f'to application {application.name}')
else:
    print('Skipped seeding — using existing production traffic.')

Spans travel through OTLP export and ingestion before they are queryable, so there is a short delay
before they show up. This cell polls until they land.

In [ ]:
import time as _time

if SEED_SYNTHETIC_SPANS:
    TIMEOUT_SECONDS = 180
    POLL_SECONDS = 10

    deadline = _time.time() + TIMEOUT_SECONDS
    seen = 0

    while _time.time() < deadline:
        probe = application.get_span_fields(
            start_time=start_time,
            end_time=datetime.now(timezone.utc),
        )
        seen = probe.total_spans
        if seen > 0:
            print(f'{seen} spans are queryable')
            break
        waited = int(TIMEOUT_SECONDS - (deadline - _time.time()))
        print(f'  waiting for ingestion... ({waited}s)')
        _time.sleep(POLL_SECONDS)

    if seen == 0:
        print(
            f'\nNo spans became queryable within {TIMEOUT_SECONDS}s.\n'
            'Ingestion latency varies by deployment. Re-run this cell to keep waiting, '
            'or continue once spans appear in the Fiddler trace explorer.'
        )

# Extend the window so freshly seeded spans are always inside it.
end_time = datetime.now(timezone.utc)
print(f'Window: {start_time.isoformat()} -> {end_time.isoformat()}')

## 2. Discover span attributes

Attribute keys vary by instrumentation, so start by asking the application what it actually emits.

`get_span_fields()` returns every attribute key seen across the time range with a **coverage
count**, plus any evaluator outputs recorded on those spans.

The count is the useful part: an attribute present on 903 of 1284 spans is a poor choice for a
required input field, because the spans missing it produce an empty string rather than an error.

In [ ]:
fields = application.get_span_fields(
    start_time=start_time,
    end_time=end_time,
)

print(f'{fields.total_spans} spans scanned (sampled={fields.sampled})\n')

if fields.total_spans == 0:
    print(
        'No spans in this window. Either:\n'
        f'  - "{application.name}" has not received traffic in the last {LOOKBACK_DAYS} days '
        '(increase LOOKBACK_DAYS), or\n'
        '  - you just created it — run the seeding section (1.5) above.'
    )

print('Span attributes:')
for attribute in sorted(fields.span_attributes, key=lambda a: -a.count):
    coverage = attribute.count / fields.total_spans if fields.total_spans else 0
    print(f'  {attribute.key:<45} {attribute.count:>6}  ({coverage:.0%})')

if fields.evaluator_outputs:
    print('\nEvaluator outputs:')
    for output in fields.evaluator_outputs:
        print(f'  {output.rule_name}::{output.output_name:<30} {output.count:>6}')

<div class="alert alert-info">

**Note on sampling.** `get_span_fields()` scans at most **10,000** matching spans. Above that it
sets `sampled=True` and the counts describe a sample rather than the full set. Narrow the time
range, `filter`, or `search` when you need exact coverage.

</div>

### 2.1 Narrow the scan (optional)

Both `get_span_fields()` and the span query in the next step accept the same two narrowing
arguments.

`filter` is a structured tree over span fields. Field names are **namespaced**:

| Namespace | Example | Matches |
|---|---|---|
| `Span::` | `Span::span_type`, `Span::duration`, `Span::status_code` | Reserved span columns |
| `SpanAttribute::` | `SpanAttribute::gen_ai.usage.total_tokens` | Any user-defined span attribute |
| `Evaluator::` | `Evaluator::Safety::Is Toxic` | An evaluator rule's output |

**To find out what you can filter on**, see
[Span and Resource Attributes](https://docs.fiddler.ai/integrations/agentic-ai/attributes) — the
full catalog of indexed attributes, which namespace each belongs to, and its value type. Step 2
above prints what *your* application actually emits; that page explains how to address each one in a
filter.

`search` is a case-insensitive substring match over span content, AND-ed with `filter`. It matches
the content attributes Fiddler full-text indexes, such as `gen_ai.llm.input.user` and
`gen_ai.llm.output`.

Limits: 3 levels of nesting, 10 rules total, and a search query of 3–64 characters.
`Span::duration` is in **nanoseconds**.

The operators available on `QueryRule` come from `OperatorType`: `EQUAL`, `NOT_EQUAL`, `IN`,
`NOT_IN`, `LESS`, `LESS_OR_EQUAL`, `GREATER`, `GREATER_OR_EQUAL`, `BETWEEN`, `NOT_BETWEEN`,
`CONTAINS`, `NOT_CONTAINS`, `BEGINS_WITH`, `ENDS_WITH`, `IS_EMPTY`, `IS_NOT_EMPTY`, `IS_NULL`,
`IS_NOT_NULL`, and `ANY`.

In [ ]:
# Example: only LLM spans slower than 2 seconds, mentioning "refund" in the input.
#
# If you seeded synthetic spans in section 1.5, this matches the slow "refund"
# conversations. Against your own traffic, expect 0 results until you adjust the
# criteria to something your spans actually satisfy -- 0 here means "nothing
# matched", not "the query failed".
slow_llm_filter = QueryCondition(
    rules=[
        QueryRule(
            field='Span::span_type',
            operator=OperatorType.EQUAL,
            value='llm',
        ),
        QueryRule(
            field='Span::duration',
            operator=OperatorType.GREATER,
            value=2_000_000_000,  # 2 seconds, in nanoseconds
        ),
    ]
)

narrowed = application.get_span_fields(
    start_time=start_time,
    end_time=end_time,
    filter=slow_llm_filter,
    search=SearchFilter(query='refund', scope=SearchScope.INPUT),
)

print(f'{narrowed.total_spans} spans match the narrowed criteria (sampled={narrowed.sampled})')

## 3. Select the spans to promote

`add_items_from_spans()` takes explicit `(trace_id, span_id)` pairs, so choose the spans first.
The helper below calls the `POST /v3/spans/query` endpoint.

It takes the same `filter` and `search` arguments as `get_span_fields()` and returns spans
newest-first. Its `page_size` maximum of **200** matches the per-call span limit on
`add_items_from_spans()`, so one page maps cleanly to one call.

In [ ]:
def query_spans(
    url,
    token,
    application_id,
    start_time,
    end_time,
    filter_=None,
    search=None,
    page_size=200,
    offset=0,
):
    """Return spans matching a filter, newest first.

    Thin wrapper over POST /v3/spans/query. Replace with the SDK method when one ships.
    """
    payload = {
        'application_id': str(application_id),
        'start_time': start_time.isoformat(),
        'end_time': end_time.isoformat(),
        'page_size': page_size,  # server maximum is 200
        'offset': offset,
    }
    if filter_ is not None:
        payload['filter'] = filter_.model_dump(mode='json')
    if search is not None:
        payload['search'] = search.model_dump(mode='json')

    response = requests.post(
        f'{url.rstrip("/")}/v3/spans/query',
        headers={'Authorization': f'Bearer {token}'},
        json=payload,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()['data']['items']

In [ ]:
spans = query_spans(
    url=URL,
    token=API_KEY,
    application_id=application.id,
    start_time=start_time,
    end_time=end_time,
    filter_=slow_llm_filter,
)

print(f'Found {len(spans)} spans\n')

# Peek at one span so you can see which attributes are available to map.
if spans:
    sample = spans[0]
    print(f'trace_id:   {sample["trace_id"]}')
    print(f'span_id:    {sample["span_id"]}')
    print(f'name:       {sample["name"]}')
    print(f'span_type:  {sample.get("span_type")}')
    print(f'duration:   {sample["duration"] / 1e9:.2f}s')
    print('\nspan_attributes:')
    print(json.dumps(sample.get('span_attributes', {}), indent=2)[:1200])

In [ ]:
# add_items_from_spans() accepts SpanReference objects or plain dicts.
span_refs = [
    SpanReference(trace_id=span['trace_id'], span_id=span['span_id'])
    for span in spans
]

print(f'Selected {len(span_refs)} spans to promote')

## 4. Create the dataset and define a field mapping

Spans and dataset items have different shapes. A span is a flat bag of OpenTelemetry attributes; a
dataset item has four named buckets. Promoting spans is therefore a **projection**, and you supply
that projection as a `FieldMapping`.

What each bucket is for when promoting production spans:

- **`inputs`** — what your task replays. Your task function receives this bucket as its `inputs`
  argument, so `inputs['user_query']` is how it reads the captured query.
- **`expected_outputs`** — the reference to compare against. Promoting the production response here
  captures "what we shipped last time," which is what makes the dataset a regression baseline.
- **`metadata`** / **`extras`** — context you want to slice results by, but that the task does not
  consume.

Read each entry as *"populate the dataset field on the left from the span attribute on the right."*
A span missing a mapped attribute yields an empty string rather than failing the request.

In [ ]:
dataset = Dataset.get_or_create(
    name=DATASET_NAME,
    application_id=application.id,
    description='Slow LLM responses promoted from production for replay',
)

print(f'Dataset: {dataset.name} ({dataset.id})')

If the dataset already has items, inspect its schema first and reuse the existing field names.
Consistency matters because experiment tasks bind to inputs **by name**.

In [ ]:
schema = dataset.get_schema()
print(f'{schema.total_items} existing items (sampled={schema.sampled})')

for bucket_name in ('inputs', 'expected_outputs', 'metadata', 'extras'):
    bucket = getattr(schema, bucket_name)
    if bucket:
        print(f'\n{bucket_name}:')
        for field in bucket:
            print(f'  {field.key:<30} {field.data_type:<10} ({field.count})')

<div class="alert alert-info">

**Use Fiddler's canonical field names** — `user_query`, `rag_response`, `retrieved_documents` —
rather than generic ones like `question` and `answer`.

The names matter most on the *task output* side: built-in evaluators bind by their score function's
parameter names, so a task returning `{'rag_response': ...}` wires into `AnswerRelevance()` and
`RAGFaithfulness()` with no `score_fn_kwargs_mapping`. Naming your dataset `inputs` to match keeps
the whole chain readable.

</div>

**Update the attribute keys below to match what Step 2 printed for your application.**

In [ ]:
mapping = FieldMapping(
    inputs={
        # gen_ai.llm.input.user is what set_input() writes, and what most
        # instrumentations emit for the user prompt.
        'user_query': 'gen_ai.llm.input.user',
    },
    expected_outputs={
        'expected_response': 'gen_ai.llm.output',
    },
    metadata={
        'model': 'gen_ai.request.model',
    },
)

print(json.dumps(mapping.model_dump(), indent=2))

## 5. Add the spans as dataset items

Fiddler resolves each span server-side from the IDs you send, applies the mapping, and writes one
dataset item per span. Span *content* never round-trips through your client, and the application is
derived from the dataset — not from your request — so a mapping cannot pull attributes out of an
application you do not have access to.

`start_time` and `end_time` must cover the spans you reference; Fiddler uses them to bound the
lookup, and a span outside the window is treated as missing.

The call is **all-or-nothing**. If any span cannot be resolved, Fiddler writes nothing and returns
`422` with a `SpanNotFound` entry per unresolved span. That makes retries safe: a failed call leaves
the dataset untouched, so you can fix the references and call again without creating duplicates.

In [ ]:
MAX_SPANS_PER_CALL = 200  # server limit

batch = span_refs[:MAX_SPANS_PER_CALL]

result = dataset.add_items_from_spans(
    spans=batch,
    mapping=mapping,
    start_time=start_time,
    end_time=end_time,
)

print(f'Created {result.items_created} dataset items')
for item_id in result.item_ids[:5]:
    print(f'  {item_id}')

### Promoting more than 200 spans

Page through your span query and call `add_items_from_spans()` once per batch. Each call is
independent, so a batch that fails does **not** roll back batches that already succeeded.

Other limits worth knowing:

| Limit | Value |
|---|---|
| Spans per call | 200 |
| Items per dataset | 10,000 |
| Spans scanned for field discovery | 10,000 |
| Span query page size | 200 |

The per-call span limit and the per-dataset item limit are deployment-configurable
(`EVALS_MAX_ADD_SPANS_TO_DATASET`, `EVALS_MAX_DATASET_ITEM_COUNT`).

In [ ]:
def promote_in_batches(dataset, span_refs, mapping, start_time, end_time, batch_size=200):
    """Promote span references in server-sized batches. Returns total items created."""
    total = 0
    for offset in range(0, len(span_refs), batch_size):
        chunk = span_refs[offset:offset + batch_size]
        result = dataset.add_items_from_spans(
            spans=chunk,
            mapping=mapping,
            start_time=start_time,
            end_time=end_time,
        )
        total += result.items_created
        print(f'  batch {offset // batch_size + 1}: +{result.items_created} items')
    return total


# Uncomment to promote everything you selected:
# total = promote_in_batches(dataset, span_refs, mapping, start_time, end_time)
# print(f'Total items created: {total}')

## 6. Verify the golden dataset

Re-read the schema to confirm the promoted fields landed where you expected.

In [ ]:
schema = dataset.get_schema()
print(f'{schema.total_items} items in "{dataset.name}"\n')

for bucket_name in ('inputs', 'expected_outputs', 'metadata', 'extras'):
    bucket = getattr(schema, bucket_name)
    if bucket:
        print(f'{bucket_name}:')
        for field in bucket:
            print(f'  {field.key:<30} {field.data_type:<10} ({field.count})')
        print()

items = list(dataset.get_items())[:3]
for item in items:
    print(f'--- item {item.id} ---')
    print(f'  inputs:           {item.inputs}')
    print(f'  expected_outputs: {item.expected_outputs}')

## 7. Run an experiment against it

The dataset is now a normal Fiddler Experiments dataset. Point a task at it and every future change
gets replayed against the traffic you promoted.

In [ ]:
from fiddler_evals import evaluate
from fiddler_evals.evaluators import AnswerRelevance


def my_agent(inputs, extras, metadata):
    """Replace with a call into your real application."""
    user_query = inputs.get('user_query', '')
    return {'rag_response': f'A fresh answer to: {user_query}'}


# Uncomment to run:
# experiment = evaluate(
#     name='golden-dataset-baseline',
#     dataset=dataset,
#     task=my_agent,
#     evaluators=[AnswerRelevance()],
# )
# print(f'Experiment: {experiment.name} ({experiment.id}) -> {experiment.status}')

## Congratulations!

You've built a golden dataset from real production traffic:

- **Discovered** which attribute keys your spans actually carry, with coverage counts
- **Queried** production spans with a structured filter
- **Mapped** span attributes onto dataset fields
- **Promoted** those spans into a reusable golden dataset
- **Replayed** it in an experiment

### Doing this in the UI

The same workflow is available without code. In the trace explorer, select spans and choose
**Add to Dataset**. The dialog walks you through choosing a dataset, mapping fields, and reviewing
the result before anything is written.

The UI additionally saves your field mapping back onto the dataset so the next run pre-fills it.
`add_items_from_spans()` does not persist the mapping — pass it explicitly on every call, or keep it
in version control alongside your evaluation code.

### Next steps

- [Golden Datasets documentation](https://docs.fiddler.ai/evaluate-and-test/golden-datasets)
- [Span and Resource Attributes](https://docs.fiddler.ai/integrations/agentic-ai/attributes) — the full catalog of filterable fields
- [Query spans REST reference](https://docs.fiddler.ai/sdk-api/rest-api/spans/query-spans)
- [Capture traces during experiments](https://docs.fiddler.ai/evaluate-and-test/eval-trace-capture)
- [Evals SDK Quick Start](https://docs.fiddler.ai/evaluate-and-test/evals-sdk-quick-start)

### Questions?

Reach out to us at [help@fiddler.ai](mailto:help@fiddler.ai).